In [1]:
from pynq.overlays.base import BaseOverlay
# Try this specific import:
from pynq.lib import MicroblazeLibrary
base = BaseOverlay("base.bit")
# Initialize the PMODB interface
lib = MicroblazeLibrary(base.PMODB, ['uart'])
# Open UART at 9600 baud (HM-10 default)
uart = lib.uart_open(0, 1) 

print("UART Initialized. Sending AT command...")
uart.write(b"AT")

AttributeError: 'MicroblazeLibrary' object has no attribute 'uart_open'

SerialException: [Errno 2] could not open port /dev/ttyPS1: [Errno 2] No such file or directory: '/dev/ttyPS1'

In [4]:
# Access the AXI GPIO block (used for GPS enable/reset)
gpio = MMIO(ol.axi_gpio_0.mmio.base_addr, 0x10000, debug=False)

In [1]:
# --- STEP 1: Locate UARTLite IP and map MMIO ---
import time

UART_IP_NAME = "gps_uart"   # confirmed earlier
UART_RANGE = 0x100          # 256 bytes is plenty

uart_base_addr = ol.ip_dict[UART_IP_NAME]["phys_addr"]
uart = MMIO(uart_base_addr, UART_RANGE, debug=False)

print(f"✔ UARTLite '{UART_IP_NAME}' mapped at 0x{uart_base_addr:08X}")


NameError: name 'ol' is not defined

In [5]:
# --- STEP 2: UARTLite register offsets and bits ---
RX_FIFO     = 0x00
TX_FIFO     = 0x04
STATUS_REG  = 0x08
CTRL_REG    = 0x0C

# STATUS bits
RX_VALID = 1 << 0
TX_EMPTY = 1 << 2
TX_FULL  = 1 << 3

# CTRL bits
RST_TX = 1 << 0
RST_RX = 1 << 1

In [7]:
# --- STEP 3: Reset UART FIFOs ---
uart.write(CTRL_REG, RST_TX)
sleep(0.05)
uart.write(CTRL_REG, RST_RX)
sleep(0.05)

print("✔ UART TX/RX FIFOs reset")
print("STATUS:", hex(uart.read(STATUS_REG)))


✔ UART TX/RX FIFOs reset
STATUS: 0x4


In [8]:
# show the keys in the overlay IP dictionary
import pprint
pprint.pprint(list(ol.ip_dict.keys()))


['axi_gpio_0',
 'axi_iic_0',
 'lora',
 'accel',
 'ACL2',
 'pwm',
 'axi_timer_0',
 'front_side',
 'driver_side',
 'passenger_side',
 'pasfront',
 'drifront',
 'pasback',
 'drivback',
 'gps_uart',
 'Bluetooth']
